In [ ]:
!pip install faster-whisper demucs

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

In [ ]:
import os
!python3 -m demucs --two-stems=vocals --mp3 "{audio_path}"
track_name = os.path.splitext(os.path.basename(audio_path))[0]
vocals_path = f"separated/htdemucs/{track_name}/vocals.mp3"
print(f"Vocals at: {vocals_path}")

In [ ]:
from faster_whisper import WhisperModel

def transcribe(vocals_path):
    print("Loading Whisper Model...")
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    print("Transcribing...")
    segments, info = model.transcribe(vocals_path, language="pa")

    result = []
    for segment in segments:
        print(f"[{int(segment.start // 60):02d}:{int(segment.start % 60):02d}] {segment.text.strip()}")
        result.append(segment)

    return result

result = transcribe(vocals_path)

In [ ]:
from google.colab import files

output_path = f"{track_name}_lyrics.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for segment in result:
        start = segment.start
        text = segment.text.strip()
        minutes = int(start // 60)
        seconds = int(start % 60)
        f.write(f"[{minutes:02d}:{seconds:02d}] {text}\n")

print(f"Lyrics saved to {output_path}")
files.download(output_path)